# Evaluate on temporal split data
* Data / labels downloaded from opentargets - 02-2022 release.
* Note: Only positives are added over time.
* We will train a new model on the 2022 data/labels, and get it's prediction on our 2025 data. We will see the ranks of any newly added clinical links!

Historical data download instructions:
* `wget --recursive --no-parent --no-host-directories --cut-dirs=8 ftp://ftp.ebi.ac.uk/pub/databases/opentargets/platform/22.02/output/etl/parquet/associationByOverallDirect/`
    * + rename `associationByOverallDirect` to `association_overall_direct`
 
* and get `known_drugs` (just to be safe) - /known_drug - they only have `knownDrugsAggregated` there.

    * assuming similar, rename to known_drug 

In [ ]:
import os
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split, GroupKFold, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    RepeatedStratifiedKFold,
    StratifiedGroupKFold,
)

from utils import (
    disease_mean_baseline,
    summarize_group_ranking_metrics,
    target_mean_baseline,
)

%load_ext autoreload
%autoreload 2
try:
    from dl_model_def import make_fs, TwoTowerDual, build_two_tower_model
    import tensorflow as tf, keras
except:
    print("No TF in env")

In [ ]:
# DATA_DIR = "./opentargets/"
DATA_DIR = "../data/opentargets/"
HISTORIC_DATA_DIR = "../data/historical_ot/22_02/"  # associationByOverallDirect/

In [ ]:
def make_ds(df: pd.DataFrame):
    feats = {
        "query": {
            "disease_text": df["disease_text"],
            "diseaseId": df["diseaseId"],
        },
        "candidate": {
            "target_text": df["target_text"],
            "targetId": df["targetId"],
        },
    }
    y = {
        "cls": df["label"].astype("float32"),  # -– 0 / 1
        "score": df["score"].astype("float32"),  # -– continuous
    }
    return tf.data.Dataset.from_tensor_slices((feats, y))


import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
)


def run_evaluation(model, test_ds, df_test, threshold=0.5):
    """
    Runs model predictions on test_ds, calculates metrics against df_test['label'],
    and prints a structured report.
    """
    print(f"\n{'=' * 40}")
    print(f"STARTING EVALUATION ON {len(df_test)} SAMPLES")

    # 1. Generate Predictions
    #    Note: test_ds must be batched and NOT shuffled to match df_test order
    print("-> Generating predictions...")
    results = model.predict(test_ds, verbose=1)

    # Extract probabilities (assuming model returns dict with key 'cls')
    # Use .ravel() to flatten (N, 1) -> (N,)
    y_pred_probs = results["cls"].ravel()

    # 2. Get Ground Truth
    y_true = df_test["label"].values.astype(int)

    # 3. Calculate Global Metrics
    roc_auc = roc_auc_score(y_true, y_pred_probs)
    pr_auc = average_precision_score(y_true, y_pred_probs)

    # 4. Generate Classification Report
    y_pred_binary = (y_pred_probs > threshold).astype(int)
    cls_report = classification_report(y_true, y_pred_binary)

    # 5. Pretty Print
    print(f"\n{'-' * 20}")
    print(f"GLOBAL METRICS (Probabilities)")
    print(f"{'-' * 20}")
    print(f"ROC-AUC : {roc_auc:.5f}")
    print(f"PR-AUC  : {pr_auc:.5f}")

    print(f"\n{'-' * 20}")
    print(f"CLASSIFICATION REPORT (Threshold: {threshold})")
    print(cls_report)
    print(f"{'=' * 20}\n")

In [ ]:
# disease_path = "../data/proc/disease_df.parquet"
# target_path = "../data/proc/target_df.parquet"

disease_path = "./copy_proc/disease_df.parquet"
target_path = "./copy_proc/target_df.parquet"

In [ ]:
# df_learn = pd.read_parquet("../data/proc/df_learn.parquet")
# print(df_learn.shape)
# display(df_learn)
disease_df = pd.read_parquet(disease_path)
print(disease_df.shape)
display(disease_df.head(2))
target_df = pd.read_parquet(target_path)
print(target_df.shape)
display(target_df.head(2))

warning - some disease ids are different? 
e.g. //ebi.ac.uk/efo/EFO_0600064	

In [ ]:
class Config:
    # DATA_DIR = "../data/opentargets/" # modern data
    DATA_DIR = HISTORIC_DATA_DIR  # "../data/historical_ot/22_02/"#
    # PROCESSED_DATA_FILE = 'final_df.parquet'
    PROCESSED_DATA_FILE = "history_df.parquet"


### newly moved up here,outside the "main":
# ## Always run this section; so we can have its filtered diseases in data (for filtering candidate diseases):
### following section extracted; used to filter out diseases (e.g. "measurement")
ta_map = disease_df.set_index("diseaseId")["name"].to_dict()
labels_to_remove = [
    "measurement",
    "phenotype",
    "biological process",
    "cell proliferation disorder",
]
exploded_tas = (
    disease_df[["diseaseId", "therapeuticAreas"]].explode("therapeuticAreas").dropna()
)
exploded_tas["label"] = exploded_tas["therapeuticAreas"].map(ta_map)
ids_to_remove = exploded_tas[exploded_tas["label"].isin(labels_to_remove)][
    "diseaseId"
].unique()


def make_target_data():
    """Orchestrates the data processing and feature generation pipeline."""
    print("--- Starting Data and Feature Pre-computation ---")

    if not os.path.exists(Config.PROCESSED_DATA_FILE):
        print(f"-> '{Config.PROCESSED_DATA_FILE}' not found. Processing raw data...")
        # disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease'))
        associations_df = pd.read_parquet(
            os.path.join(Config.DATA_DIR, "association_overall_direct")
        )  # association_overall_direct - modern. Renamed folder to that
        known_drug_df = pd.read_parquet(os.path.join(Config.DATA_DIR, "known_drug"))

        associations_filtered = associations_df[
            ~associations_df["diseaseId"].isin(ids_to_remove)
        ]
        ###### Should we be dropping phase - na cases? (dropna)?? (Also later on) ######
        validated_targets = known_drug_df.dropna(subset=["phase"])["targetId"].unique()
        working_df = associations_filtered[
            associations_filtered["targetId"].isin(validated_targets)
        ].copy()

        pairs_with_evidence = known_drug_df.dropna(subset=["phase"])[
            ["targetId", "diseaseId"]
        ].drop_duplicates()
        pairs_with_evidence["label"] = 1

        final_df = pd.merge(
            working_df, pairs_with_evidence, on=["targetId", "diseaseId"], how="left"
        )
        final_df["label"] = final_df["label"].fillna(0).astype(int)
        print(final_df["label"].describe().round(3))
        final_df.to_parquet(Config.PROCESSED_DATA_FILE)
        print(f"-> Saved processed data to '{Config.PROCESSED_DATA_FILE}'.")

    else:
        print(f"-> Found '{Config.PROCESSED_DATA_FILE}'. Skipping raw data processing.")
        final_df = pd.read_parquet(Config.PROCESSED_DATA_FILE)
    return final_df

In [ ]:
final_df = make_target_data().drop(columns="evidenceCount", errors="ignore")
print(final_df.shape[0])
## try to clean some diseaseIds to better match future
final_df["diseaseId"] = (
    final_df["diseaseId"].str.split("/").str[-1]
)  # no real improvement
## filter for matching idsease ID - warning! many rows dropped!!

final_df = final_df.query("diseaseId in @disease_df.diseaseId")
print(
    final_df.shape[0], "# after matching on disease records (beware data naming changes"
)

final_df = final_df.query("targetId in @target_df.targetId")
print(
    final_df.shape[0], "# after matching on target records (beware data naming changes"
)

print(final_df.label.agg(["mean", "sum"]).round(2))
final_df

In [ ]:
print("# unique positives in past data:")
final_df.query("label>0").nunique()

#current (2025) release data and labels (made in regular notebook)

In [ ]:
df_future = pd.read_parquet("final_df.parquet")  # previously made,
print(df_future.label.agg(["mean", "sum"]).round(2))
df_future

In [ ]:
print("# unique positives in Future data:")
df_future.query("label>0").nunique()  # ~150 more targets and 400 more diseases.

In [ ]:
print(df_future.shape[0])  # 663351
print(df_future.query("diseaseId in @final_df.diseaseId").shape[0])  # 492711
print(df_future.query("targetId in @final_df.targetId").shape[0])  # 612676

I have a historical snapshot of a dataset (final_df ). (Open Targets disease-drug associations). I am making a temporal split using data from the modern version of the data (final_df). 
The target is if a diseaseId, targetId have a positive association (label =1). 
In data updates, only positive cases were added. 

I want to do an anti join between the final_df and df_future , so that we will have only rows/instances left from df_future that did not appear in final_df .  (Then I can train a model on final_df  , and look at the predictions in df_future , e.g. average rank or similar). 

Note that an instance just at the disease and target id level may appear in the past, but if added (with label=1) in future, then that's the very thing we want kept for the test!


###### The Anti-Join
* This performs a left join on the composite key (diseaseId, targetId, label). By including label, instances that existed in final_df as 0 but appear in df_future as 1 will fail to match (good) and will be kept in the result as left_only.

In [ ]:
# Define keys including label to capture state changes (0 -> 1)
join_keys = ["diseaseId", "targetId", "label"]

# Perform anti-join: Keep future rows that don't have an exact match (ID+Label) in history
df_test = (
    df_future.merge(final_df[join_keys], on=join_keys, how="left", indicator=True)
    .query('_merge == "left_only"')
    .drop(columns="_merge")
)

### add past score - useful for baseline
# Merge ONLY the score from the past, matching on IDs
df_test = df_test.merge(
    final_df[["diseaseId", "targetId", "score"]],
    on=["diseaseId", "targetId"],
    how="left",
    suffixes=("", "_past"),
)
print("0 score cases in past, pre impute", (df_test["score_past"] == 0).sum())
# Fill NaNs with 0 (if a row didn't exist in the past, its score was effectively 0)
df_test["score_past"] = df_test["score_past"].fillna(0)
print("0 score cases in past", (df_test["score_past"] == 0).sum())
print(df_test.label.agg(["mean", "sum"]).round(2))
display(df_test)

In [ ]:
# Check 1: Strict exclusion verification
# Ensure absolutely no rows in df_test exist in final_df with the same label
overlaps = df_test.merge(final_df[join_keys], on=join_keys, how="inner")
assert len(overlaps) == 0, f"Found {len(overlaps)} rows in Test that exist in History!"

# Check 2: Logic verification (0 -> 1 Transition)
# Find a pair that was 0 in history but 1 in future (if any exist)
past_neg = final_df[final_df.label == 0][["diseaseId", "targetId"]]
fut_pos = df_future[df_future.label == 1][["diseaseId", "targetId"]]

# Identify candidates that changed from 0 to 1
converts = past_neg.merge(fut_pos, on=["diseaseId", "targetId"], how="inner")

if not converts.empty:
    # Pick a sample ID pair that changed 0->1
    sample = converts.iloc[0]

    # Assert this specific converted pair is present in our new df_test
    is_present = (
        df_test[
            (df_test.diseaseId == sample.diseaseId)
            & (df_test.targetId == sample.targetId)
            & (df_test.label == 1)
        ].shape[0]
        > 0
    )

    assert is_present, (
        "Logic Error: Known 0->1 transitions were excluded from the test set!"
    )

print(f"Sanity checks passed. Test set size: {len(df_test)}")

In [ ]:
def merge_df_dis_target(df):
    s1 = df.shape[0]
    print(s1)
    df = df.merge(disease_df[["diseaseId", "disease_text_embed"]], on="diseaseId")
    df = df.merge(target_df[["targetId", "target_text_embed"]], on="targetId")
    df.rename(
        columns={
            "disease_text_embed": "disease_text",
            "target_text_embed": "target_text",
        },
        inplace=True,
        errors="ignore",
    )
    assert s1 == df.shape[0], s1 - df.shape[0]
    return df


final_df = merge_df_dis_target(final_df)
df_test = merge_df_dis_target(df_test)

In [ ]:
print(final_df.nunique())  # 6489 diseases, 1406 targets
print(final_df.label.mean())

In [ ]:
print(df_test.nunique())  # 11624 diseases (twice as many!), 1521 targets
print(df_test.label.mean())

### Past score baseline
* without the fillna, corr is actually negative
* 

In [ ]:
print("Past score corr with future:")
print(df_test.label.corr(df_test["score_past"]).round(3))  # 0.06
print("Past score rocauc on future:")
print(
    f"ROC-AUC : {roc_auc_score(y_true=df_test.label, y_score=df_test['score_past']):.5f}"
)  ## 55 rocauc
print(f"PR-AUC  : {average_precision_score(df_test.label, df_test['score_past']):.5f}")

### Historical mean baselines

In [ ]:
y_historic_target_mean = target_mean_baseline(test_df=df_test, full_df=final_df)
print("Historical target mean baseline")
print(
    f"ROC-AUC : {roc_auc_score(y_true=df_test.label, y_score=y_historic_target_mean):.5f}"
)
print(f"PR-AUC  : {average_precision_score(df_test.label, y_historic_target_mean):.5f}")

y_historic_disease_mean = disease_mean_baseline(test_df=df_test, full_df=final_df)
print("\nHistorical disease mean baseline")
print(
    f"ROC-AUC : {roc_auc_score(y_true=df_test.label, y_score=y_historic_disease_mean):.5f}"
)
print(
    f"PR-AUC  : {average_precision_score(df_test.label, y_historic_disease_mean):.5f}"
)

In [ ]:
# Global prior baseline (single scalar = train label mean)
from sklearn.metrics import roc_auc_score, average_precision_score

# Use the train split to compute the prior
global_prior = train_df["label"].mean()
print("Global prior (train mean):", round(global_prior, 4))
y_prior = np.full(len(df_test), global_prior)
print(f"ROC-AUC : {roc_auc_score(df_test.label, y_prior):.5f}")
print(f"PR-AUC  : {average_precision_score(df_test.label, y_prior):.5f}")

In [ ]:
# Fast temporal baselines: Matrix Factorization + TF-IDF cosine
from baselines.run_baselines import BaselineConfig, fit_mf_predict, fit_tfidf_predict

fast_cfg = BaselineConfig(models=("mf", "tfidf"))
train_fast = final_df[
    ["diseaseId", "targetId", "label", "disease_text", "target_text"]
].copy()
test_fast = df_test[
    ["diseaseId", "targetId", "label", "disease_text", "target_text"]
].copy()

y_true = test_fast["label"].to_numpy()

mf_temporal_scores = fit_mf_predict(
    train_df=train_fast, test_df=test_fast, config=fast_cfg
)
tfidf_temporal_scores = fit_tfidf_predict(
    train_df=train_fast, test_df=test_fast, config=fast_cfg
)

print("Temporal Matrix Factorization baseline")
print(f"ROC-AUC : {roc_auc_score(y_true, mf_temporal_scores):.5f}")
print(f"PR-AUC  : {average_precision_score(y_true, mf_temporal_scores):.5f}")

print("\nTemporal TF-IDF cosine baseline")
print(f"ROC-AUC : {roc_auc_score(y_true, tfidf_temporal_scores):.5f}")
print(f"PR-AUC  : {average_precision_score(y_true, tfidf_temporal_scores):.5f}")

temporal_fast_baselines = pd.DataFrame(
    {
        "Model": ["Matrix Factorization", "TF-IDF cosine"],
        "ROC-AUC": [
            roc_auc_score(y_true, mf_temporal_scores),
            roc_auc_score(y_true, tfidf_temporal_scores),
        ],
        "PR-AUC": [
            average_precision_score(y_true, mf_temporal_scores),
            average_precision_score(y_true, tfidf_temporal_scores),
        ],
    }
)
display(temporal_fast_baselines.round(5))

## Train DL model

In [ ]:
# ---------------------------------------------------------
# 1)  make a **target–hold-out** split  (70 % / 30 %)
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

# unique targets → split the *IDs* (not the rows)
train_tids, test_tids = train_test_split(
    final_df["targetId"].unique(),
    test_size=0.01,
    random_state=42,
    shuffle=True,
    # stratify = final_df.drop_duplicates(subset=["targetId"])["label"]
)

train_df = final_df[final_df["targetId"].isin(train_tids)].copy()
val_df = final_df[final_df["targetId"].isin(test_tids)].copy()

* warning: is the vocab adapted for both? 

In [ ]:
model = build_two_tower_model(final_df)  # df_learn vs train_df

In [ ]:
TEMPORAL_AUX_SCORE_WEIGHT = 0.1  # set to 0.0 for an auxiliary-free temporal ablation

losses = {
    "cls": keras.losses.BinaryCrossentropy(from_logits=False),
    # "cls"  : keras.losses.BinaryFocalCrossentropy(apply_class_balancing=False),
    "score": keras.losses.MeanSquaredError(),  # or MAE / Huber
}

loss_weights = {"cls": 1.0, "score": TEMPORAL_AUX_SCORE_WEIGHT}

metrics = {
    "cls": [
        keras.metrics.AUC(name="auc"),
        keras.metrics.AUC(curve="PR", name="pr_auc"),
    ],
    "score": [keras.metrics.RootMeanSquaredError(name="rmse")],
}

model.compile(
    optimizer=keras.optimizers.Adam(7e-3),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

train_ds = make_ds(train_df).shuffle(2_00_000).batch(512).prefetch(tf.data.AUTOTUNE)
val_ds = make_ds(val_df).batch(2048).prefetch(tf.data.AUTOTUNE)

test_ds = (
    make_ds(df_test.filter(train_df.columns.to_list(), axis=1))
    .batch(1024)
    .prefetch(tf.data.AUTOTUNE)
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        "val_cls_loss", mode="min", factor=0.2, patience=1
    ),
    keras.callbacks.EarlyStopping(
        "val_cls_loss", mode="min", patience=2, restore_best_weights=False
    ),
]


In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=callbacks)

### Model results in future prediction:

In [ ]:
run_evaluation(model, test_ds, df_test)

In [ ]:
temporal_predictions = df_test[["diseaseId", "targetId", "label", "score_past"]].copy()
temporal_predictions["otrec_pred"] = model.predict(test_ds, verbose=1)["cls"].ravel()
temporal_predictions["target_mean_pred"] = target_mean_baseline(
    test_df=df_test, full_df=final_df
)
temporal_predictions["disease_mean_pred"] = disease_mean_baseline(
    test_df=df_test, full_df=final_df
)

for name, score_col in [
    ("OTRec", "otrec_pred"),
    ("Historical OTP score", "score_past"),
    ("Historical target mean", "target_mean_pred"),
    ("Historical disease mean", "disease_mean_pred"),
]:
    _, summary = summarize_group_ranking_metrics(
        temporal_predictions,
        score_col=score_col,
        group_col="diseaseId",
        ks=(1, 5, 10),
        min_positives=1,
    )
    print(f"\n{name}")
    display(summary.to_frame(name="value").T.round(4))

OTTree temporal baseline
* Train catboost (tree) model on same set of features and future evaluation setup

In [ ]:
# Train a CatBoost model on `final_df` and evaluate on `df_test`
# Uses text features: disease_text, target_text and categorical: diseaseId.
try:
    from catboost import CatBoostClassifier, Pool
except Exception as e:
    print("CatBoost not available in this environment:", e)
    raise
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
)

# Choose features to use (keep it small and compatible)
features = ["disease_text", "target_text", "diseaseId"]  # , 'score_past'
# Filter to columns that actually exist in the dataframes
features = [f for f in features if f in final_df.columns]
print("Using features:", features)

text_features = [f for f in ["disease_text", "target_text"] if f in features]
cat_features = [f for f in ["diseaseId"] if f in features]

# Build Pools
train_pool = Pool(
    data=final_df[features],
    label=final_df["label"],
    text_features=text_features,
    cat_features=cat_features,
)
test_pool = Pool(
    data=df_test[features],
    label=df_test["label"],
    text_features=text_features,
    cat_features=cat_features,
)

# Train model (keep params conservative; adjust later if desired)
model_cb = CatBoostClassifier(
    depth=8, device_config="gpu", eval_metric="AUC", random_seed=42, verbose=100
)
model_cb.fit(train_pool)

# Predict and evaluate
y_pred_proba = model_cb.predict_proba(test_pool)[:, 1]
print("ROC-AUC :", round(roc_auc_score(df_test["label"], y_pred_proba), 5))
print("PR-AUC  :", round(average_precision_score(df_test["label"], y_pred_proba), 5))

y_pred = (y_pred_proba > 0.5).astype(int)
print("Classification report (threshold=0.5):")
print(classification_report(df_test["label"], y_pred))